# VisClick — Colab setup

Run cells **top to bottom**. Authorise Google Drive when prompted.

- **GitHub** → code at `/content/visclick`
- **Drive** `MyDrive/visclick` → datasets + weights (persists after disconnect)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
if not os.path.isdir("/content/visclick"):
    subprocess.run(["git", "clone", REPO, "/content/visclick"], check=True)
else:
    subprocess.run("cd /content/visclick && git pull --rebase", shell=True, check=False)
%cd /content/visclick


In [ ]:
import subprocess
try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
        print(subprocess.check_output(["nvidia-smi"]).decode())
except Exception as e:
    print("GPU check:", e)


In [ ]:
import os
DRIVE = "/content/drive/MyDrive/visclick"
os.makedirs(DRIVE, exist_ok=True)
for sub in (
    "data/raw", "data/clay", "data/vins", "data/webui", "data/ui_vision",
    "data/unified", "data/source_train", "data/desktop_labeled", "data/desktop_test",
    "weights/baseline_source", "weights/experiments", "videos",
):
    os.makedirs(os.path.join(DRIVE, sub), exist_ok=True)
print("DRIVE =", DRIVE)


## 00 — Data acquisition

Saves to **Drive** `visclick/data/`. RICO may need a manual browser download — see cell comments.


In [ ]:
!pip install -q ultralytics==8.* opencv-python "huggingface_hub[cli]>=0.20" tqdm matplotlib pandas


In [ ]:
# --- A) CLAY labels (images need RICO separately) ---
import os, subprocess
DRIVE = "/content/drive/MyDrive/visclick"
os.chdir(os.path.join(DRIVE, "data", "raw"))
if not os.path.isdir("clay"):
    subprocess.run(["git", "clone", "https://github.com/google-research-datasets/clay.git", "clay"], check=True)
import shutil
os.makedirs(os.path.join(DRIVE, "data", "clay"), exist_ok=True)
for f in os.listdir("clay"):
    src = os.path.join("clay", f)
    dst = os.path.join(DRIVE, "data", "clay", f)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)
print("CLAY repo files copied to", os.path.join(DRIVE, "data", "clay"))


In [ ]:
# --- B) UI-Vision (desktop benchmark, ~781 MB) ---
import os, subprocess
DRIVE = "/content/drive/MyDrive/visclick"
out = os.path.join(DRIVE, "data", "ui_vision")
# huggingface-cli
subprocess.run(
    f'pip install -q huggingface_hub && hf download ServiceNow/ui-vision --repo-type dataset --local-dir "{out}"',
    shell=True,
    check=False,
)
print("UI-Vision target:", out)


In [ ]:
# --- C) Zenodo unified RICO+WebUI (large). Uncomment wget/unzip. ---
# In Colab, %cd is a magic command (OK on its own line).
DRIVE = "/content/drive/MyDrive/visclick"
%cd /content/drive/MyDrive/visclick/data/raw
# !wget -c -O rico_webui_unified.zip "https://zenodo.org/records/19195885/files/dataset.zip?download=1"
# !unzip -q -o rico_webui_unified.zip -d ../unified
print("If you uncommented: data is in data/unified. Verify URL on Zenodo if download fails.")


In [ ]:
# --- D) VINS (clone repo; image archives from VINS README) ---
DRIVE = "/content/drive/MyDrive/visclick"
%cd /content/drive/MyDrive/visclick/data/raw
# !git clone https://github.com/sbunian/VINS.git vins_repo
print("Extract VINS images to", f"{DRIVE}/data/vins", "per official README")


**RICO screenshots:** `interactionmining.org/rico` — often no stable wget. Download the zip, upload to Drive `data/raw/rico_unique_uis.zip`, then in a new cell: `!unzip -q rico_unique_uis.zip -d ../rico` from `data/raw`.

**Next:** assemble `data/source_train` in YOLO format (see detailed plan C.5) or use `unified` if you downloaded Zenodo.
